In [ ]:
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)
file_path = DATA_DIR / "DMI_API_3stations.csv"

In [ ]:
BASE_URL = "https://opendataapi.dmi.dk/v2/climateData/collections/stationValue/items"
parameters = ["mean_radiation"]
stations = ["06188", "06174", "06156"] #06188 = Sjaelsmark, 06174 = Tessebolle, 06156 = Holbaek Flygeplads
# List to collect DataFrames
dfs = []
for col in parameters:
    for station in stations:
        # Parameters
        params = {
            "stationId": station,                     # your station (ROSKILDE LUFTHAVN)
            "parameterId": col,                       # parameter to retrieve
            "timeResolution": "hour",                 # hourly values
            "datetime": "2025-02-01T00:00:00Z/2026-02-28T23:59:59Z",
            "limit": 30000 ,                           # max records
        }
        # Send request
        response = requests.get(BASE_URL, params=params)
        # Check success
        if response.status_code == 200:
            data = response.json()
            # Each record is a GeoJSON feature
            features = data.get("features", [])
            # Parse all stationValue fields into a list of dicts
            records = []
            for f in features:
                props = f.get("properties", {})
                geom = f.get("geometry", {})
                coords = geom.get("coordinates", None)
                # Include geometry if you want
                record = {
                    **props,
                    "longitude": coords[0] if coords else None,
                    "latitude": coords[1] if coords else None
                }
                records.append(record)
            # Convert to DataFrame and add to list if not empty
            if records:
                df_param = pd.DataFrame(records)
                dfs.append(df_param)
        else:
            print("Error", response.status_code, response.text)
# Stack all DataFrames on top of each other
if dfs:
    df_stacked = pd.concat(dfs, ignore_index=True)
else:
    print("No valid data returned for any parameter")
df_stacked.to_csv(file_path)

In [ ]:
def format_DMI_data_radation(file_path,stations):
    try:
        df = pd.read_csv(file_path,delimiter=",",decimal=".")
    except Exception as e:
        print(f"Error reading file: {e}")
        return
    df = df[df["validity"]] # want only valid columns 

    df = df.rename(columns={"from":"ts"}) # rename columns
    df["ts"] = pd.to_datetime(df['ts']) # setting date time


    df = df.pivot_table(
    index='ts', 
    columns='stationId', 
    values='value', 
    aggfunc='first'  
)
    
    df.index = df.index.strftime('%Y-%m-%d %H:%M:%S')

    df.to_csv(DATA_DIR /"DMI_radiation.csv") 

In [ ]:
stations = ["6188", "6174", "6156"]
format_DMI_data_radation(file_path,stations)